# Feature Importance (SHAP)

## Set Up

In [ ]:
import sys, os

sys.path.append(os.path.abspath("../"))
from src.config import BASE_PATH, SEED

from src.swarm import run_swarms_shap

In [ ]:
MODEL_LIST = ["xgb", "lgbm", "nn", "stack", "lr"]
OUTCOME_LIST = [
    "SERIOUS",
    "ANY",
    "PNEUMO",
    "CARDIAC_COMP",
    "VTE",
    "SEPSIS",
    "SSI",
    "UTI",
    "RENAL",
    "UNPLNREOP",
    "MORT",
]

FEAT_ORDER = [
    # demographics
    "AGE",
    "BMI",
    "RACE",
    # pre-op health + comorbidities
    "DIABETES",
    "SMOKE",
    "HXCOPD",
    "HXCHF",
    "HYPERMED",
    "DISCANCR",
    "BLEEDDIS",
    "ASACLAS",
    # blood vals
    "PRALBUM",
    "PRWBC",
    "PRHCT",
    "PRPLATE",
    # intra-op
    "INOUT",
    "URGENCY",
    "SURGSPEC",
    "ANESTHES",
    ## CPT Op ##
    # resection
    "PARTIALCPT",
    "SUBSIMPLECPT",
    "MODIFIEDRADICALCPT",
    # axillary
    "SNLBCPT",
    "ALNDCPT",
    # impact-based
    "IMMEDIATECPT",
    "TEINSERTIONCPT",
    # autologous
    "FREECPT",
    "SINTRAMCPT",
    # adjunct + revision
    "BREASTREDCPT",
    "ADJTISTRANSCPT",
]

Ensure we included all col in feat_order var

In [ ]:
from src.data_utils import get_data

outcome_dict = get_data("ANY", file_dir=BASE_PATH / "data" / "processed")
dummy_df = outcome_dict["X_test"][:5]
all_cols = set()
for col in dummy_df.columns:
    col_split = col.split("_")
    if len(col_split) == 1:
        all_cols.add(col)
    else:
        col_name = col_split[0]
        all_cols.add(col_name)
assert set(FEAT_ORDER) == set(all_cols)

## RUN SHAP

In [ ]:
def create_cmd_str(
    model_name,
    outcome_name,
    feat_order,
    model_path,
    data_dir,
    result_path,
    n_background,
    seed,
):
    feat_list_str = ",".join(feat_order)
    cmd_str = f"export PYTHONPATH={BASE_PATH};\
                uv run python -m src.feat_imp \
                --model_name {model_name}\
                --outcome_name {outcome_name}\
                --feat_order '{feat_list_str}'\
                --model_path {model_path}\
                --data_dir {data_dir}\
                --seed {seed}\
                --result_path {result_path}"
    if n_background != None:
        cmd_str += f" --n_background {n_background}"
    return " ".join(cmd_str.split())

In [ ]:
SWARM_DIR = BASE_PATH / "swarm/shap"

model_config_dict = {
    "xgb": {
        "swarm_time": "0:10:00",
        "gb": 2,
        "partition": "quick",
        "batch_size": 11,
    },
    "lgbm": {
        "swarm_time": "0:10:00",
        "gb": 2,
        "partition": "quick",
        "batch_size": 11,
    },
    "nn": {
        "swarm_time": "20:00:00",
        "gb": 2,
        "partition": "norm",
        "batch_size": 1,
    },
    "stack": {
        "swarm_time": "4-00:00:00",
        "gb": 2,
        "partition": "norm",
        "batch_size": 1,
    },
    "lr": {
        "swarm_time": "0:05:00",
        "gb": 2,
        "partition": "quick",
        "batch_size": 11,
    },
}

In [ ]:
full_cmd_list = []
cmd_dir = SWARM_DIR / "commands"
data_base_path = BASE_PATH / "data" / "processed"
for model_abrv in MODEL_LIST:
    swarm_path = cmd_dir / f"{model_abrv}.swarm"
    swarm_path.parent.mkdir(exist_ok=True, parents=True)
    if swarm_path.exists():
        swarm_path.unlink()
    # Less background data for KernelExplainer
    if model_abrv in ["stack", "nn"]:
        n_background = 100
    else:
        n_background = None
    model_cmd_list = []
    for outcome in OUTCOME_LIST:
        # SHAP on uncalibrated models
        model_base_path = BASE_PATH / "models" / "trained" / outcome
        result_path = BASE_PATH / "results" / "shap" / outcome / f"{model_abrv}.xlsx"
        if model_abrv == "nn":
            model_path = model_base_path / "nn.pt"
        else:
            model_path = model_base_path / f"{model_abrv}.joblib"
        cmd_str = create_cmd_str(
            model_name=model_abrv,
            outcome_name=outcome,
            feat_order=FEAT_ORDER,
            model_path=model_path,
            data_dir=data_base_path,
            result_path=result_path,
            n_background=n_background,
            seed=SEED,
        )
        model_cmd_list.append(cmd_str)
    swarm_path.write_text("\n".join(model_cmd_list))
    full_cmd_list += model_cmd_list
print(f"Num commands: {len(full_cmd_list)}")

Run sequentially

In [ ]:
# import subprocess

# for iteration, cmd in enumerate(full_cmd_list):
#     print(f"{iteration +1}/{len(full_cmd_list)}...")
#     subprocess.run(cmd, shell=True, check=True)
#     break

Run swarms

In [ ]:
run_swarms_shap(
    swarm_log_dir=SWARM_DIR / "logs",
    cmd_dir=SWARM_DIR / "commands",
    model_list=MODEL_LIST,
    config_dict=model_config_dict,
)

## Aggregate tables

- One table per outcome
- Append each model's MASV col 
- Also a quick heatmap of MASVs for each outcome

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from functools import reduce
from src.data_utils import export_data


def make_heatmap(combined_df, outcome_name):
    # Set Feature as index, drop any all-NaN rows
    heatmap_df = combined_df.set_index("Feature").dropna(how="all")

    fig, ax = plt.subplots(
        figsize=(len(heatmap_df.columns) * 1.5, len(heatmap_df) * 0.4)
    )

    sns.heatmap(
        heatmap_df,
        ax=ax,
        annot=True,
        fmt=".3f",
        cmap="YlOrRd",  # good for MASV (all positive, higher = more important)
        linewidths=0.5,
        linecolor="lightgrey",
        cbar_kws={"label": "Relative MASV"},
        annot_kws={"size": 8},
    )

    ax.set_title(outcome_name, fontsize=13, pad=12)
    ax.set_xlabel("Model", fontsize=10)
    ax.set_ylabel("Feature", fontsize=10)
    ax.tick_params(axis="x", rotation=45, labelsize=9)
    ax.tick_params(axis="y", rotation=0, labelsize=8)

    plt.tight_layout()
    # plt.savefig(export_dir / f"{outcome_name}_heatmap.png", dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
res_dir = BASE_PATH / "results/shap"
export_dir = BASE_PATH / "results/shap/combined"
combined_list = []
for outcome_dir in res_dir.iterdir():
    model_dfs = []
    outcome_name = outcome_dir.name
    if outcome_name == "combined":
        continue
    for model_path in outcome_dir.iterdir():
        model_name = model_path.stem
        shap_df = pd.read_excel(model_path, index_col=0)
        # Rename MASV col to model name
        shap_df.rename(columns={"Relative_ MASV": model_name}, inplace=True)
        shap_df = shap_df[["Feature", model_name]]
        model_dfs.append(shap_df)
    # Combine
    combined_df = reduce(
        lambda left, right: pd.merge(left, right, on="Feature", how="outer"), model_dfs
    )
    ## Fix order
    combined_df = combined_df.set_index("Feature").reindex(FEAT_ORDER).reset_index()
    # Plot
    make_heatmap(combined_df, outcome_name)
    # Export
    export_data(
        data_to_export=combined_df, export_path=export_dir / f"{outcome_name}.xlsx"
    )